# Model Creation Pipeline
This notebook handles final feature engineering and data splitting before training our models.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('../Data/cleaned_assignment_data.csv')
print(f'Original Data Shape: {df.shape}')


Original Data Shape: (2572, 45)


### 1. Drop Identifiers

In [2]:
identifiers = ['ID', 'country_id', 'application_id', 'product_id', 'customer_id']
df = df.drop(columns=identifiers, errors='ignore')
print(f'Shape after dropping identifiers: {df.shape}')


Shape after dropping identifiers: (2572, 41)


### 2. Feature Engineering (Dates)
Convert datetime strings to numerical differences (e.g. days late).

In [3]:
# Convert to datetime objects
date_cols = ['due_date', 'paid_date', 'arrived_date', 'first_status_day_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Calculate days late (positive means paid after due date)
if 'paid_date' in df.columns and 'due_date' in df.columns:
    df['days_late'] = (df['paid_date'] - df['due_date']).dt.days

# Calculate days to arrive
if 'arrived_date' in df.columns and 'paid_date' in df.columns:
    df['days_to_arrive'] = (df['arrived_date'] - df['paid_date']).dt.days

# Drop raw date columns (Tree models can't read them)
df = df.drop(columns=date_cols, errors='ignore')

# Check if first_status_time_of_day exists and drop it too (it's a time string)
if 'first_status_time_of_day' in df.columns:
    df = df.drop(columns=['first_status_time_of_day'], errors='ignore')

print('New numerical date features created. Raw dates dropped.')


New numerical date features created. Raw dates dropped.


### 3. Encoding Categoricals (One-Hot Encoding)
Convert string categories to binary 1s and 0s using pd.get_dummies.

In [4]:
categorical_cols = ['Variable_5', 'Variable_6', 'Variable_12', 'Variable_13', 'Variable_14', 'Variable_45']
# Filter out any that were already dropped
categorical_cols = [col for col in categorical_cols if col in df.columns]

# One-hot encode
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dummy_na=True)
print(f'Shape after One-Hot Encoding: {df.shape}')


Shape after One-Hot Encoding: (2572, 162)


### 4. Data Splitting
Extract the prediction set, then perform an 80/20 train/test split.

In [5]:
# The target is NA for the prediction set
prediction_set = df[df['Target'].isna()].copy()
train_val_set = df[df['Target'].notna()].copy()

# The Target column is loaded as string/float if it had NAs, let's make sure it's numeric for the training set
train_val_set['Target'] = train_val_set['Target'].astype(int)

# Drop Target from prediction set (it's just NaNs anyway)
X_pred = prediction_set.drop(columns=['Target'])

# Prepare X and y for training
X = train_val_set.drop(columns=['Target'])
y = train_val_set['Target']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Prediction Set Shape: {X_pred.shape}')
print(f'Training Set Shape: {X_train.shape}')
print(f'Testing Set Shape: {X_test.shape}')


Prediction Set Shape: (515, 161)
Training Set Shape: (1645, 161)
Testing Set Shape: (412, 161)


### 5. Export Final Datasets
Save the perfectly cleaned and split datasets into a new folder for easy loading during model training.

In [6]:
import os
# Create a new directory to store the final ready-to-model data
os.makedirs('final_model_data', exist_ok=True)

# Save training and testing sets
X_train.to_csv('../Data/final_model_data/X_train.csv', index=False)
X_test.to_csv('../Data/final_model_data/X_test.csv', index=False)
y_train.to_csv('../Data/final_model_data/y_train.csv', index=False)
y_test.to_csv('../Data/final_model_data/y_test.csv', index=False)

# Save the final prediction set (the rows we need to predict at the very end)
X_pred.to_csv('../Data/final_model_data/X_pred.csv', index=False)

print('Successfully saved all 5 datasets to the ../Data/final_model_data/ folder.')

Successfully saved all 5 datasets to the ../Data/final_model_data/ folder.
